In [48]:
"""
Kaggle Road Vehicle Images Dataset 转换为YOLOv8格式脚本
功能：准备数据集、处理预划分的训练/测试集、转换标签格式、创建配置文件
"""

import os
import shutil
import yaml
from pathlib import Path
import glob

In [49]:
# 1. 配置参数
# 您的YOLOv8项目的数据集根目录 (新数据存放的地方)
dataset_root = Path(r"D:\\yolov8\\论文\\yolov8\\datasets\\road_vehicle_dataset")
# Kaggle原始数据解压后的目录
raw_dataset_dir = dataset_root / "road-vehicle-images-dataset" / "trafic_data"

# 2704 训练, 304 测试
train_images_src = raw_dataset_dir / "train" / "images"
train_labels_src = raw_dataset_dir / "train" / "labels"

# 原始测试集在 'valid' 文件夹中
test_images_src = raw_dataset_dir / "valid" / "images"
test_labels_src = raw_dataset_dir / "valid" / "labels"
# --- 修改结束 ---

# 16个类别
user_classes = [
    'bicycle', 'bus', 'car', 'minibus', 'minivan', 'motorbike',
    'pickup', 'policecar', 'rickshaw', 'scooter', 'suv', 'taxi',
    'three wheelers -CNG-', 'truck', 'van', 'wheelbarrow'
]

In [50]:
# 2. 创建目录结构 (YOLOv8 最终需要的目录)
def create_directory_structure():
    train_images_dir = dataset_root / "images" / "train"
    test_images_dir = dataset_root / "images" / "test"  # 我们用 'test' 目录作为验证集
    train_labels_dir = dataset_root / "labels" / "train"
    test_labels_dir = dataset_root / "labels" / "test"

    for dir_path in [train_images_dir, test_images_dir, train_labels_dir, test_labels_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)

    return train_images_dir, test_images_dir, train_labels_dir, test_labels_dir

In [51]:
# 3. 获取文件列表
def get_image_files(src_dir):
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_files = []
    if not src_dir.exists():
        print(f"!!! 严重错误：目录不存在: {src_dir}")
        print("!!! 请再次确认您的Kaggle原始数据集路径是否正确。")
        return []

    for ext in image_extensions:
        image_files.extend(glob.glob(str(src_dir / f"*{ext}"), recursive=False))
    return image_files

In [52]:
# 4. 获取类别映射
def get_class_mapping():
    # 读取原始yaml文件获取类别信息
    original_yaml_path = raw_dataset_dir / "data_1.yaml"
    if original_yaml_path.exists():
        with open(original_yaml_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        original_classes = data.get('names', [])

        # 创建原始类别到ID的映射
        original_to_id = {cls: idx for idx, cls in enumerate(original_classes)}

        # 创建用户要求的类别到新ID的映射
        user_to_new_id = {cls: idx for idx, cls in enumerate(user_classes)}
        original_to_new_id = {}

        print("--- 正在生成类别映射 ---")
        for user_cls in user_classes:
            if user_cls in original_to_id:
                original_to_new_id[original_to_id[user_cls]] = user_to_new_id[user_cls]
                print(f"映射类别: {user_cls} (原始ID: {original_to_id[user_cls]} -> 新ID: {user_to_new_id[user_cls]})")
            else:
                print(f"警告: 类别 '{user_cls}' 在原始数据集中未找到")

        return original_to_new_id
    else:
        print(f"!!! 错误: 未找到原始YAML文件: {original_yaml_path}")
        return None

In [53]:
# 5. 查找标注文件路径
def find_label_file(image_path, label_src_dir):
    img_name = os.path.splitext(os.path.basename(image_path))[0]
    label_file = label_src_dir / f"{img_name}.txt"

    if label_file.exists():
        return str(label_file)
    else:
        # 备用检查：万一Kaggle的label和image在同一个文件夹
        alt_label_file = Path(image_path).with_suffix('.txt')
        if alt_label_file.exists():
            return str(alt_label_file)

    return None

In [54]:
# 6. 转换标签格式
def convert_labels(image_files, label_src_dir, labels_dest_dir, class_mapping):
    total_converted = 0
    skipped_no_label = 0
    skipped_no_class = 0

    if not label_src_dir.exists():
        print(f"!!! 严重错误：原始标签目录不存在: {label_src_dir}")
        print("!!! 请再次确认您的Kaggle原始数据集路径是否正确。")
        return

    for img_path in image_files:
        try:
            img_name = os.path.splitext(os.path.basename(img_path))[0]

            # 查找标注文件
            label_file = find_label_file(img_path, label_src_dir)

            if label_file:
                with open(label_file, 'r', encoding='utf-8') as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        original_class_id = int(parts[0])
                        # 检查是否需要映射这个类别
                        if original_class_id in class_mapping:
                            new_class_id = class_mapping[original_class_id]
                            new_line = f"{new_class_id} {' '.join(parts[1:])}\n"
                            new_lines.append(new_line)

                if new_lines:
                    output_path = labels_dest_dir / f"{img_name}.txt"
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.writelines(new_lines)
                    total_converted += 1
                else:
                    skipped_no_class += 1
            else:
                skipped_no_label += 1

        except Exception as e:
            print(f"处理图像 {img_path} 时出错: {e}")

    print(f"标签转换完成：")
    print(f"- 成功转换 (至少有一个有效框): {total_converted}")
    print(f"- 跳过 (未找到对应.txt): {skipped_no_label}")
    print(f"- 跳过 (框内类别均非目标): {skipped_no_class}")

In [55]:
# 7. 移动(复制)图像文件
def move_images(image_files, dest_dir):
    success_count = 0

    for img_path in image_files:
        try:
            img_name = os.path.basename(img_path)
            dst = dest_dir / img_name
            if os.path.exists(img_path):
                shutil.copy(img_path, dst)
                success_count += 1
        except Exception as e:
            print(f"复制 {img_path} 时出错: {e}")

    print(f"图像复制完成：\n- 成功: {success_count}")

In [57]:
# 8. 创建YAML配置文件
def create_yaml_config(train_images_path, val_images_path):
    # 路径转换为YOLOv8期望的 / 分隔符格式
    train_path_str = str(train_images_path).replace('\\', '/')
    val_path_str = str(val_images_path).replace('\\', '/')

    yaml_content = f"""# 训练集和验证集（测试集）的路径
# 此文件由脚本自动生成
train: {train_path_str}
val: {val_path_str}

# 类别数量
nc: 16

# 类别名称 (与论文 [cite: 81] 一致)
names:
  - bicycle
  - bus
  - car
  - minibus
  - minivan
  - motorbike
  - pickup
  - policecar
  - rickshaw
  - scooter
  - suv
  - taxi
  - 'three wheelers -CNG-'
  - truck
  - van
  - wheelbarrow
"""

    yaml_path = dataset_root / "my_road_data.yaml"
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(yaml_content)

    print(f"YAML配置文件已创建：{yaml_path}")
    return yaml_path

In [58]:
# --- 主函数 ---
def main():
    print("开始准备Kaggle Road Vehicle Images Dataset...")

    # 1. 创建目标目录结构
    train_images_dir, test_images_dir, train_labels_dir, test_labels_dir = create_directory_structure()

    # 2. 获取类别映射
    class_mapping = get_class_mapping()
    if not class_mapping:
        print("错误：未能生成类别映射，请检查 data_1.yaml 文件路径。程序终止。")
        return

    # 3. 获取原始文件列表
    print("\n--- 正在定位原始文件 ---")
    train_images = get_image_files(train_images_src)
    test_images = get_image_files(test_images_src) # 已修改为 'valid' 路径

    print(f"\n--- 检查文件数量 ---")
    print(f"找到 {len(train_images)} 张原始训练图像。 ")
    print(f"找到 {len(test_images)} 张原始测试图像。 ")

    if len(train_images) == 2704 and len(test_images) == 300:
        print("文件数量与论文  一致，很好。")
    else:
        print("!!! 警告：找到的图像数量与论文不符。")
        print("!!! 请务必检查 'train_images_src' 和 'test_images_src' 路径是否正确。")
        print("!!! 否则无法复现论文结果。")


    # 4. 移动(复制)图像文件
    print("\n--- 正在复制图像 ---")
    print("正在复制训练集图像...")
    move_images(train_images, train_images_dir)
    print("正在复制测试集图像...")
    move_images(test_images, test_images_dir)

    # 5. 转换标签格式
    print("\n--- 正在转换标签 ---")
    print("正在转换训练集标签...")
    convert_labels(train_images, train_labels_src, train_labels_dir, class_mapping)
    print("正在转换测试集标签...")
    convert_labels(test_images, test_labels_src, test_labels_dir, class_mapping)

    # 6. 创建YAML配置文件
    print("\n--- 正在创建配置文件 ---")
    yaml_path = create_yaml_config(
        train_images_dir,
        test_images_dir
    )

    print("\n数据集准备完成！")
    print(f"您现在可以使用此 data.yaml 文件运行： {yaml_path}")
    print("请使用源码中的 'train.py' 和 'mycfg' 目录下的 .yaml 配置文件进行训练。")

if __name__ == "__main__":
    main()

开始准备Kaggle Road Vehicle Images Dataset...
--- 正在生成类别映射 ---
映射类别: bicycle (原始ID: 3 -> 新ID: 0)
映射类别: bus (原始ID: 4 -> 新ID: 1)
映射类别: car (原始ID: 5 -> 新ID: 2)
映射类别: minibus (原始ID: 8 -> 新ID: 3)
映射类别: minivan (原始ID: 9 -> 新ID: 4)
映射类别: motorbike (原始ID: 10 -> 新ID: 5)
映射类别: pickup (原始ID: 11 -> 新ID: 6)
映射类别: policecar (原始ID: 12 -> 新ID: 7)
映射类别: rickshaw (原始ID: 13 -> 新ID: 8)
映射类别: scooter (原始ID: 14 -> 新ID: 9)
映射类别: suv (原始ID: 15 -> 新ID: 10)
映射类别: taxi (原始ID: 16 -> 新ID: 11)
映射类别: three wheelers -CNG- (原始ID: 17 -> 新ID: 12)
映射类别: truck (原始ID: 18 -> 新ID: 13)
映射类别: van (原始ID: 19 -> 新ID: 14)
映射类别: wheelbarrow (原始ID: 20 -> 新ID: 15)

--- 正在定位原始文件 ---

--- 检查文件数量 ---
找到 2704 张原始训练图像。 (论文要求: 2704)
找到 300 张原始测试图像。 (论文要求: 304)
!!! 警告：找到的图像数量与论文不符。
!!! 请务必检查 'train_images_src' 和 'test_images_src' 路径是否正确。
!!! 否则无法复现论文结果。

--- 正在复制图像 ---
正在复制训练集图像...
图像复制完成：
- 成功: 2704
正在复制测试集图像...
图像复制完成：
- 成功: 300

--- 正在转换标签 ---
正在转换训练集标签...
标签转换完成：
- 成功转换 (至少有一个有效框): 2696
- 跳过 (未找到对应.txt): 0
- 跳过 (框内类别均非目标): 8
正在转换测试集标签...
标签